# 🎬 AI Video Dubber v5.0 — Production Pipeline

**Professional AI dubbing** — translates and dubs Chinese/Asian drama videos into 18+ languages.

### 🆓 100% FREE — No Paid APIs

| Component | Runs On | Cost |
|-----------|---------|------|
| 🐟 **Fish Speech TTS** | Colab T4 GPU (local) | **FREE** |
| 🎵 **Demucs** (vocal separation) | Colab T4 GPU (local) | **FREE** |
| 🧠 **Groq AI** (translate/direct) | Groq Cloud (free tier) | **FREE** |
| 🎤 **Whisper** (transcribe) | Groq Cloud (free tier) | **FREE** |
| 💾 **Storage** | Google Drive | **FREE** |

### ✨ v5.0 Features
- 🐟 Fish Speech running **locally on GPU** — no TTS API key needed
- 🎵 Demucs removes Chinese vocals, keeps clean background music
- 🎤 Smart voice matching — native-sounding regional voices
- 🎭 Emotion-aware TTS — angry scenes sound angry
- 📊 EBU R128 professional loudness normalization

### ⚡ Requirements
- **T4 GPU**: Runtime → Change runtime type → T4 GPU
- **Groq API Key**: Free at https://console.groq.com/keys

---

## Step 1: Setup Environment & GPU

In [ ]:
# ⚡ First: Enable GPU → Runtime → Change runtime type → T4 GPU

import subprocess, os

# Check GPU
gpu_check = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True)
if gpu_check.returncode == 0:
    print(f'✅ GPU: {gpu_check.stdout.strip()}')
else:
    print('⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/DubbedVideos'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'✅ Google Drive mounted! Output: {DRIVE_OUTPUT}')

## Step 2: Install Dependencies

In [ ]:
# Install core dependencies
!pip install -q groq>=0.11.0 edge-tts>=6.1.0 yt-dlp indic-transliteration>=2.3.0 nest-asyncio>=1.5.0
!pip install -q demucs httpx torchaudio

# Apply nest_asyncio fix for Colab
import nest_asyncio
nest_asyncio.apply()

# Verify
!ffmpeg -version 2>&1 | head -1
!yt-dlp --version
print('\n✅ Core dependencies installed!')

## Step 3: Install Fish Speech (LOCAL GPU TTS — No API Key!)

This installs Fish Speech to run **directly on the Colab T4 GPU**.
No cloud API, no API key, no cost. The model runs right here.

In [ ]:
%%time
import os, subprocess, time

FISH_DIR = '/content/fish-speech'
MODEL_DIR = '/content/fish-speech-model'

# Step 1: Clone Fish Speech
if not os.path.exists(FISH_DIR):
    print('📦 Cloning Fish Speech...')
    !git clone https://github.com/fishaudio/fish-speech.git {FISH_DIR}
else:
    print('✅ Fish Speech already cloned')

# Step 2: Install Fish Speech
print('📦 Installing Fish Speech...')
!cd {FISH_DIR} && pip install -q -e . 2>&1 | tail -3

# Step 3: Download model from HuggingFace (Fish Speech 1.5 — fits on T4)
if not os.path.exists(MODEL_DIR):
    print('📥 Downloading Fish Speech model (~2GB)...')
    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='fishaudio/fish-speech-1.5',
        local_dir=MODEL_DIR,
        ignore_patterns=['*.md', '*.txt', 'samples/*']
    )
    print(f'✅ Model downloaded to {MODEL_DIR}')
else:
    print('✅ Model already downloaded')

# Show model files
model_files = os.listdir(MODEL_DIR) if os.path.exists(MODEL_DIR) else []
print(f'\n📂 Model files: {model_files}')
print('\n✅ Fish Speech ready for local GPU inference!')

In [ ]:
# Start Fish Speech local API server in the background
import subprocess, time, os, signal

FISH_DIR = '/content/fish-speech'
MODEL_DIR = '/content/fish-speech-model'

# Kill any existing server
!pkill -f 'fish_speech' 2>/dev/null || true
time.sleep(1)

print('🐟 Starting Fish Speech local server on port 8080...')
print('   (This runs the TTS model on the T4 GPU — no API key needed!)\n')

# Start the API server
# Fish Speech supports multiple server modes — try the available one
server_cmd = None

# Try the tools/api_server approach (Fish Speech 1.x)
api_server_path = os.path.join(FISH_DIR, 'tools', 'api_server.py')
serve_module_check = subprocess.run(
    ['python', '-c', 'import fish_speech; print(dir(fish_speech))'],
    capture_output=True, text=True, cwd=FISH_DIR
)

if os.path.exists(api_server_path):
    server_cmd = f'cd {FISH_DIR} && python tools/api_server.py --listen 0.0.0.0 --port 8080 --checkpoint-path {MODEL_DIR}'
    print(f'Using: tools/api_server.py')
else:
    # Try module-based approach
    server_cmd = f'cd {FISH_DIR} && python -m fish_speech.serve --listen 0.0.0.0 --port 8080 --checkpoint-path {MODEL_DIR}'
    print(f'Using: fish_speech.serve module')

# Start in background
server_proc = subprocess.Popen(
    server_cmd, shell=True,
    stdout=open('/content/fish_server.log', 'w'),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid
)

# Wait for server to start
print('⏳ Waiting for server to load model...')
import httpx
for i in range(60):  # Wait up to 60 seconds
    time.sleep(2)
    try:
        r = httpx.get('http://localhost:8080/', timeout=2.0)
        if r.status_code < 500:
            print(f'\n✅ Fish Speech server running on localhost:8080!')
            print(f'   🐟 Model loaded on GPU — ready for TTS generation')
            print(f'   🆓 No API key needed!')
            break
    except:
        if i % 5 == 0:
            print(f'   Loading... ({i*2}s)')
else:
    print('\n⚠️ Fish Speech server did not start in time.')
    print('   Checking logs...')
    !tail -20 /content/fish_server.log
    print('\n💡 The pipeline will still work using Edge TTS as fallback.')
    print('   Edge TTS gives decent quality but Fish Speech is better.')

## Step 4: Clone Dubber Agent & Set Groq Key

In [ ]:
# Clone the dubber repo
REPO_DIR = '/content/chinese-drama-dubber'
if os.path.exists(REPO_DIR):
    import shutil
    shutil.rmtree(REPO_DIR)
!git clone https://github.com/Dipesh600/chinese-drama-dubber.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'📁 Working directory: {os.getcwd()}')
print(f'\n✅ Repository cloned!')

In [ ]:
# Set Groq API Key (the ONLY key you need — it's free)
# Get yours at: https://console.groq.com/keys

try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('✅ Groq API key loaded from Colab Secrets!')
except:
    import getpass
    os.environ['GROQ_API_KEY'] = getpass.getpass('Enter your Groq API Key: ')
    print('✅ Groq API key set!')

# Verify
key = os.environ.get('GROQ_API_KEY', '')
print(f'🔑 Groq Key: {key[:8]}...{key[-4:]}' if len(key) > 12 else '❌ No key set!')

# Check Fish Speech status
try:
    import httpx
    r = httpx.get('http://localhost:8080/', timeout=2.0)
    print(f'🐟 Fish Speech: ✅ Running locally on GPU (no API key)')
except:
    print(f'🔊 Fish Speech: Not running — will use Edge TTS (still works fine)')

## Step 5: 🎬 Dub Your Video!

### Supported Languages
Hindi, Nepali, Tamil, Telugu, Bengali, Marathi, Gujarati, Kannada, Malayalam, Urdu, English, Spanish, French, Portuguese, German, Japanese, Korean, Arabic, Turkish

In [ ]:
#@title 🎬 Dubbing Configuration { display-mode: "form" }

#@markdown ### Video Settings
VIDEO_URL = 'https://youtu.be/m_mJhZuM4Bk' #@param {type:"string"}
TARGET_LANGUAGE = 'Hindi' #@param ['Hindi', 'Nepali', 'Tamil', 'Telugu', 'Bengali', 'Marathi', 'Gujarati', 'Kannada', 'Malayalam', 'Urdu', 'English', 'Spanish', 'French', 'Portuguese', 'German', 'Japanese', 'Korean', 'Arabic', 'Turkish']
SOURCE_LANGUAGE = 'zh' #@param ['zh', 'en', 'ja', 'ko', 'auto']

#@markdown ### Audio Settings
KEEP_BACKGROUND_MUSIC = True #@param {type:"boolean"}
USE_DEMUCS = True #@param {type:"boolean"}
USE_FISH_SPEECH = True #@param {type:"boolean"}

#@markdown ### Description
VIDEO_DESCRIPTION = 'Chinese drama video' #@param {type:"string"}

# Check what TTS engine is available
fish_running = False
try:
    import httpx
    r = httpx.get('http://localhost:8080/', timeout=2.0)
    fish_running = r.status_code < 500
except:
    pass

print(f'🎬 Video:      {VIDEO_URL}')
print(f'🌐 Language:   {TARGET_LANGUAGE}')
print(f'🎵 BG Music:   {"Demucs + Smart Ducking" if USE_DEMUCS else "Smart Ducking"}')
print(f'🔊 TTS Engine: {"🐟 Fish Speech LOCAL (GPU)" if fish_running and USE_FISH_SPEECH else "Edge TTS"}')
print(f'📁 Output:     {DRIVE_OUTPUT}')
print(f'\n⏳ Starting v5.0 pipeline...')
print('=' * 70)

In [ ]:
# Run the v5.0 dubbing pipeline!
import sys, importlib
sys.path.insert(0, REPO_DIR)

# Force reload all modules
for mod_name in list(sys.modules.keys()):
    if mod_name in ['orchestrator', 'director', 'preprocessor', 'translator',
                    'dialogue_writer', 'sentence_merger', 'voice_caster',
                    'voice_catalog', 'tts_engine', 'assembler', 'romanizer',
                    'validator', 'audio_separator']:
        del sys.modules[mod_name]

from orchestrator import run_agent

result = run_agent(
    url=VIDEO_URL,
    target_lang=TARGET_LANGUAGE,
    source_lang=SOURCE_LANGUAGE,
    user_description=VIDEO_DESCRIPTION,
    output_dir=DRIVE_OUTPUT,
    preserve_bg=KEEP_BACKGROUND_MUSIC,
    use_fish_audio=USE_FISH_SPEECH,
    use_demucs=USE_DEMUCS
)

if result['success']:
    print('\n' + '=' * 70)
    print(f'  ✅ DUBBING COMPLETE!')
    print(f'  📹 Video:      {result["video_path"]}')
    print(f'  📝 Subtitles:  {result["srt_path"]}')
    print(f'  🎭 Content:    {result.get("content_type", "?")} | {result.get("real_speaker_count", "?")} speakers')
    print(f'  ⏱️ Time:       {result.get("processing_time", "?")}s')
    print(f'  💾 Size:       {result.get("size_mb", "?")}MB')
    print(f'  🔊 TTS:        {result.get("tts_engine", "?")}')
    print(f'  🎵 Demucs:     {"✓ Clean BG" if result.get("demucs_used") else "Not used"}')
    print(f'  🗂️ Saved to:   Google Drive → DubbedVideos/')
    print('=' * 70)
else:
    print(f'\n❌ FAILED at stage {result.get("stage", "?")}: {result.get("error", "Unknown")}')
    print(f'   Work dir: {result.get("work_dir", "?")}')
    print(f'\n💡 Pipeline has crash recovery — just re-run this cell!')

## Step 6: Preview & Download

In [ ]:
# Play the dubbed video!
if result.get('success'):
    from IPython.display import Video, display
    try:
        display(Video(result['video_path'], embed=True, width=640))
    except:
        print(f'Video at: {result["video_path"]}')

    srt_path = result.get('srt_path', '')
    if srt_path and os.path.exists(srt_path):
        print('\n📝 Subtitles:')
        with open(srt_path, encoding='utf-8') as f:
            print(''.join(f.readlines()[:15]))
else:
    print('No video to preview.')

In [ ]:
# Save clean copy to Google Drive
if result.get('success'):
    import shutil, time
    ts = time.strftime('%Y%m%d_%H%M%S')
    final_dir = os.path.join(DRIVE_OUTPUT, f'dubbed_{TARGET_LANGUAGE}_{ts}')
    os.makedirs(final_dir, exist_ok=True)

    shutil.copy2(result['video_path'], os.path.join(final_dir, f'dubbed_{TARGET_LANGUAGE}.mp4'))
    if os.path.exists(result.get('srt_path', '')):
        shutil.copy2(result['srt_path'], os.path.join(final_dir, f'subtitles_{TARGET_LANGUAGE}.srt'))

    print(f'✅ Saved to: Google Drive → DubbedVideos → dubbed_{TARGET_LANGUAGE}_{ts}/')

---
## 🔄 Dub Another Video
Go back to **Step 5**, change URL/language, and run again!

### 💡 Tips
- **T4 GPU required** for Fish Speech + Demucs
- **Only 1 API key needed**: Groq (free at console.groq.com)
- **Fish Speech** runs locally on the GPU — zero cost TTS
- **Crash recovery**: If pipeline fails, just re-run — it resumes from where it stopped
- **No Fish Speech?** Set `USE_FISH_SPEECH = False` → uses Edge TTS (still decent)